In [29]:
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras import layers

In [30]:
tf.__version__, keras.__version__

('2.20.0', '3.13.2')

### Load Data

In [31]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
print(f"{x_train.shape = }, {type(x_train)}")
print(f"{x_test.shape = }, {type(x_test)}")

x_train.shape = (60000, 28, 28), <class 'numpy.ndarray'>
x_test.shape = (10000, 28, 28), <class 'numpy.ndarray'>


### Data Preprocessing

In [32]:
def max_min_values(x_train, x_test):
    print(f"Train\n    Max: {x_train.max()}\n    Min: {x_train.min()}") 
    print(f"Test\n    Max: {x_test.max()}\n    Min: {x_test.min()}")  

max_min_values(x_train, x_test)

Train
    Max: 255
    Min: 0
Test
    Max: 255
    Min: 0


In [33]:
# Normalization (Scaling the range [0, 255] to [0.0, 1.0])
x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255

max_min_values(x_train, x_test)

Train
    Max: 1.0
    Min: 0.0
Test
    Max: 1.0
    Min: 0.0


In [34]:
# Adding channel dimension (60000, 28, 28) -> (60000, 28, 28, 1) 
print(f"Previous: {x_train.shape = }")
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)
print(f"New: {x_train.shape = }")

Previous: x_train.shape = (60000, 28, 28)
New: x_train.shape = (60000, 28, 28, 1)


In [35]:
n_classes = len(np.unique(y_train))
input_shape = x_train.shape[1:]
n_classes, input_shape

(10, (28, 28, 1))

In [36]:
y_train[:5]

array([5, 0, 4, 1, 9], dtype=uint8)

In [37]:
# convert class vectors to binary class matrices
y_train = keras.utils.to_categorical(y_train, n_classes)
y_test = keras.utils.to_categorical(y_test, n_classes)

y_train[:5]

array([[0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]])

### Build Neural Network

In [ ]:

# inputs = keras.Input(shape=input_shape)
# x = layers.Conv2D(32, kernel_size=(3, 3), activation="relu")(inputs)
# x = layers.MaxPool2D(pool_size=(2,2))(x)
# x = layers.Conv2D(64, kernel_size=(3, 3), activation="relu")(x)
# x = layers.MaxPool2D(pool_size=(2,2))(x)
# x = layers.Flatten()(x)
# x = layers.Dropout(0.5)(x)
# outputs = layers.Dense(n_classes, activation="softmax")(x)

# model = keras.Model(inputs=inputs, outputs=outputs)
# model.summary()


Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │        16,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,826 (136.04 KB)

 Trainable params: 34,826 (136.04 KB)

 Non-trainable params: 0 (0.00 B)

In [39]:
class ImageClassifier(keras.Model):
    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.conv2d_1 = layers.Conv2D(32, (3, 3), activation="relu", name="Conv1")
        self.conv2d_2 = layers.Conv2D(64, (3, 3), activation="relu", name="Conv2")
        self.max_pool_1 = layers.MaxPool2D((2, 2), name="MaxPool1")
        self.max_pool_2 = layers.MaxPool2D((2, 2), name="MaxPool2")
        self.flatten = layers.Flatten(name="Flatten")
        self.dropout = layers.Dropout(0.5, name='Dropout')
        self.outputs = layers.Dense(n_classes, activation="softmax", name="Output")

    def call(self, inputs):
        x = self.conv2d_1(inputs)
        x = self.max_pool_1(x)
        x = self.conv2d_2(x)
        x = self.max_pool_2(x)
        x = self.flatten(x)
        x = self.dropout(x)
        outputs = self.outputs(x)
        return outputs
    
    def build_graph(self):
        x = keras.Input(shape=input_shape, name="Input")
        return keras.Model(inputs=x, outputs=self.call(x), name="ImageClassifier")
    
model = ImageClassifier()
model.build_graph().summary()

Model: "ImageClassifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input (InputLayer)              │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv1 (Conv2D)                  │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPool1 (MaxPooling2D)         │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2 (Conv2D)                  │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPool2 (MaxPooling2D)         │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout (Dropout)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output (Dense)                  │ (None, 10)             │        16,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,826 (136.04 KB)

 Trainable params: 34,826 (136.04 KB)

 Non-trainable params: 0 (0.00 B)

### Train

In [ ]:
batch_size = 128
epochs = 15
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
history = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1)

Epoch 1/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 19s 39ms/step - accuracy: 0.9178 - loss: 0.2953 - val_accuracy: 0.9772 - val_loss: 0.0765
Epoch 2/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 18s 34ms/step - accuracy: 0.9783 - loss: 0.0744 - val_accuracy: 0.9838 - val_loss: 0.0564
Epoch 3/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 14s 32ms/step - accuracy: 0.9830 - loss: 0.0558 - val_accuracy: 0.9875 - val_loss: 0.0486
Epoch 4/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 14s 32ms/step - accuracy: 0.9869 - loss: 0.0431 - val_accuracy: 0.9882 - val_loss: 0.0421
Epoch 5/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 14s 33ms/step - accuracy: 0.9889 - loss: 0.0378 - val_accuracy: 0.9863 - val_loss: 0.0426
Epoch 6/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 14s 32ms/step - accuracy: 0.9902 - loss: 0.0318 - val_accuracy: 0.9895 - val_loss: 0.0367
Epoch 7/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 14s 33ms/step - accuracy: 0.9918 - loss: 0.0264 - val_accuracy: 0.9883 - val_loss: 0.0417
Epoch 8/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 14s 32ms/step - accuracy: 0.9924 - loss: 0.0248 - 

### Evaluate 

In [41]:
score = model.evaluate(x_test, y_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])

Test loss: 0.03770565986633301
Test accuracy: 0.989799976348877
